# 3장 07. Argo CD와 KServe manifest 확인

이 notebook은 Kubernetes에 MLflow tracking service를 먼저 배포한 뒤, KServe `InferenceService`로 모델 serving 후보를 배포하는 manifest 흐름을 확인합니다. 실제 sync는 Argo CD가 Git의 원하는 상태를 cluster live state로 맞추는 단계입니다.


## 기본 개념: GitOps, Argo CD, KServe

Kubernetes 기본 개념은 앞의 `05_kubernetes_concepts.ipynb`에서 이미 확인했습니다. 여기서는 배포 파일을 읽는 데 필요한 세 가지를 봅니다.

GitOps는 Git에 적힌 배포 파일을 원하는 상태로 보고, cluster가 그 상태와 같아지도록 맞추는 방식입니다. Argo CD는 Git repository의 특정 path를 보고 Kubernetes cluster에 적용합니다. 그래서 강의에서는 `kubectl apply`를 직접 반복하기보다, Argo CD `Application`이 어떤 Git path를 바라보는지 확인합니다.

Kustomize는 공통 배포 파일인 `base`와 환경별 수정값인 `overlay`를 분리하는 도구입니다. 이번 실습에서는 `gitops/base`에 공통 resource를 두고, 수강생이 바꿀 값은 `gitops/overlays/student`에 둡니다.

KServe는 Kubernetes 위에서 모델 serving endpoint를 선언하는 도구입니다. `InferenceService`에는 어떤 모델을 어떤 storage URI에서 읽을지, 어떤 predictor로 띄울지 같은 정보가 들어갑니다.

이번 배포 순서는 MLflow tracking 관련 Kubernetes resource를 먼저 확인하고, 그다음 KServe `InferenceService`를 확인하는 흐름입니다. live cluster가 없으면 sync 성공이라고 쓰지 않고, manifest inspection까지만 확인했다고 기록합니다.


In [1]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


In [2]:
from pathlib import Path

application_candidates = [
    Path("demos/ch03_docker_kubernetes/argocd/application.yaml"),
    Path("../../demos/ch03_docker_kubernetes/argocd/application.yaml"),
    Path("artifacts/deployment/chapter_03/argocd/application.yaml"),
    Path("../artifacts/deployment/chapter_03/argocd/application.yaml"),
]
inference_candidates = [
    Path("demos/ch03_docker_kubernetes/gitops/base/inferenceservice.yaml"),
    Path("../../demos/ch03_docker_kubernetes/gitops/base/inferenceservice.yaml"),
    Path("artifacts/deployment/chapter_03/gitops/base/inferenceservice.yaml"),
    Path("../artifacts/deployment/chapter_03/gitops/base/inferenceservice.yaml"),
]
ingress_patch_candidates = [
    Path("demos/ch03_docker_kubernetes/gitops/overlays/student/ingress-host-patch.yaml"),
    Path("../../demos/ch03_docker_kubernetes/gitops/overlays/student/ingress-host-patch.yaml"),
    Path("artifacts/deployment/chapter_03/gitops/overlays/student/ingress-host-patch.yaml"),
    Path("../artifacts/deployment/chapter_03/gitops/overlays/student/ingress-host-patch.yaml"),
]

application_path = next(path for path in application_candidates if path.exists())
inference_path = next(path for path in inference_candidates if path.exists())
ingress_patch_path = next(path for path in ingress_patch_candidates if path.exists())
application = application_path.read_text()
inference = inference_path.read_text()
ingress_patch = ingress_patch_path.read_text()

print("application:", application_path)
print("inferenceservice:", inference_path)
print("ingress patch:", ingress_patch_path)


application: ../../demos/ch03_docker_kubernetes/argocd/application.yaml
inferenceservice: ../../demos/ch03_docker_kubernetes/gitops/base/inferenceservice.yaml
ingress patch: ../../demos/ch03_docker_kubernetes/gitops/overlays/student/ingress-host-patch.yaml


## 1. Argo CD Application에서 Git path 확인

Argo CD는 어떤 Git 경로를 클러스터 상태로 맞출지 선언합니다. 이 경로가 잘못되면 MLflow나 KServe manifest가 있어도 cluster에 적용되지 않습니다.


In [3]:
for line in application.splitlines():
    stripped = line.strip()
    if stripped.startswith(("repoURL:", "targetRevision:", "path:", "namespace:")):
        print(stripped)

print()
print("expected_source_path=", "demos/ch03_docker_kubernetes/gitops/overlays/student" in application)
print("repo_url_is_placeholder=", "REPLACE_WITH_COURSE_GIT_REPO_URL" in application)


namespace: argocd
repoURL: REPLACE_WITH_COURSE_GIT_REPO_URL
targetRevision: HEAD
path: demos/ch03_docker_kubernetes/gitops/overlays/student
namespace: ai-quality

expected_source_path= True
repo_url_is_placeholder= True


출력에서 `path`가 `gitops/overlays/student`를 가리키면 Argo CD가 수강생용 Kustomize overlay를 바라보는 것입니다. `repoURL`이 placeholder로 보이는 것은 course repository 안에서는 정상입니다. live 환경에서는 `00_setup_argocd_gitops.sh connect`가 실행 시점에 수강생 repository URL로 바꿔 적용합니다.


## 2. 수강생별 ingress 주소 수정 지점

MLflow UI를 browser에서 열려면 ingress host가 필요합니다. 이 값은 강의장, VM, DNS, tunnel 구성에 따라 달라지므로 `base`가 아니라 수강생 overlay에서 바꿉니다.


In [4]:
print(ingress_patch.strip())
print()
print("ingress_host_needs_student_value=", "REPLACE_WITH_YOUR_INGRESS_DOMAIN" in ingress_patch)


- op: replace
  path: /spec/rules/0/host
  value: mlflow.REPLACE_WITH_YOUR_INGRESS_DOMAIN.example.com

ingress_host_needs_student_value= True


출력에 `REPLACE_WITH_YOUR_INGRESS_DOMAIN`이 남아 있으면 아직 live sync 준비가 끝난 것이 아닙니다. 수강생은 이 값을 본인의 ingress 주소로 바꾼 뒤 commit/push해야 Argo CD가 그 변경을 읽을 수 있습니다.


## 3. Argo CD와 Git repository 연결 순서

Argo CD가 Git repository를 읽으려면 repository credential과 Application이 필요합니다. 실습 script는 SSH Deploy key를 만들고, 그 key를 Argo CD repository credential로 등록한 뒤, Application을 cluster에 적용합니다.

```bash
bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh check
bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh key

# 출력된 public key를 GitHub repository Settings -> Deploy keys에 추가합니다.
# Allow write access는 체크하지 않습니다.

ARGOCD_REPO_URL=git@github.com:<your-org-or-user>/<your-repo>.git   bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh connect

bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh sync
```

이 단계는 live cluster가 있는 VM에서 진행합니다. notebook 환경에서 `argocd` CLI나 cluster가 없다면 연결 성공이라고 기록하지 않고, manifest 구조 확인까지만 기록합니다.


## 4. KServe InferenceService에서 모델 후보 확인

KServe manifest에는 serving resource와 model artifact 위치가 들어갑니다. 여기서는 `storageUri`가 앞에서 확인한 MLflow/model artifact 흐름과 이어지는지 봅니다.


In [5]:
for keyword in ["InferenceService", "risk-classifier", "candidate", "storageUri"]:
    print(keyword, "=>", keyword in inference)


InferenceService => True
risk-classifier => True
candidate => True
storageUri => True
